# 🔍 Análisis Exploratorio de Datos (EDA)
## Online Shoppers Purchasing Intention Dataset

**Objetivo**: Explorar y analizar el dataset para entender patrones de comportamiento de visitantes en e-commerce.

**Dataset**: UCI ML Repository - 12,330 sesiones de usuarios únicos  
**Target**: Revenue (TRUE/FALSE) - 84.5% / 15.5% (desbalanceado)

---

## 1️⃣ Importar Librerías

In [ ]:
# Manipulación de datos
import pandas as pd
import numpy as np

# Visualización
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Configuración de warnings y estilo
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print("✅ Librerías importadas correctamente")
print(f"📦 Pandas: {pd.__version__} | Numpy: {np.__version__}")

## 2️⃣ Cargar Dataset

In [ ]:
# Cargar datos
df = pd.read_csv('../data/01_raw/online_shoppers_intention.csv')

print(f"📊 Dataset cargado exitosamente")
print(f"📏 Dimensiones: {df.shape[0]:,} filas × {df.shape[1]} columnas")
print(f"💾 Memoria: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

## 3️⃣ Vista General de los Datos

In [ ]:
# Vista de las primeras filas para entender estructura del dataset
print("🔍 Primeras 10 filas del dataset:")
df.head(10)

In [ ]:
# Últimas filas
print("🔍 Últimas 5 filas del dataset:")
df.tail()

In [ ]:
# Información general
print("📋 Información general del dataset:")
print("="*80)
df.info()

In [ ]:
# Nombres de columnas
print("📝 Columnas del dataset:")
print("="*80)
for i, col in enumerate(df.columns, 1):
    print(f"{i:2d}. {col}")

## 4️⃣ Análisis de Tipos de Datos

Verificamos los tipos de datos y realizamos conversiones necesarias para el análisis.

In [ ]:
# Analizar tipos de datos
print("🔤 Tipos de datos:")
print("="*80)
print(df.dtypes)
print("\n📊 Resumen por tipo:")
print(df.dtypes.value_counts())

In [ ]:
# Verificar si Revenue y Weekend son booleanos
print("🔍 Valores únicos de Revenue:")
print(df['Revenue'].unique())
print(f"\nTipo actual: {df['Revenue'].dtype}")

print("\n🔍 Valores únicos de Weekend:")
print(df['Weekend'].unique())
print(f"Tipo actual: {df['Weekend'].dtype}")

In [ ]:
# Convertir Revenue y Weekend a tipo booleano para mejor manejo
# Estas variables son binarias por naturaleza
if df['Revenue'].dtype != bool:
    df['Revenue'] = df['Revenue'].astype(bool)
    print("✅ Revenue convertido a booleano")

if df['Weekend'].dtype != bool:
    df['Weekend'] = df['Weekend'].astype(bool)
    print("✅ Weekend convertido a booleano")

print("\n📊 Tipos de datos actualizados:")
print(df[['Revenue', 'Weekend']].dtypes)

## 5️⃣ Calidad de los Datos

Verificamos valores faltantes, duplicados y estadísticas descriptivas.

In [ ]:
# Verificar valores faltantes (crítico para ML)
missing = df.isnull().sum()
print(f"📊 Total de valores faltantes: {missing.sum()}")
if missing.sum() > 0:
    print("\n⚠️ Columnas con valores faltantes:")
    print(missing[missing > 0])
else:
    print("✅ No hay valores faltantes en el dataset")

In [ ]:
# Verificar duplicados (pueden sesgar el análisis)
duplicates = df.duplicated().sum()
print(f"📊 Total de filas duplicadas: {duplicates}")
if duplicates > 0:
    print(f"⚠️ {duplicates} filas duplicadas encontradas ({duplicates/len(df)*100:.2f}%)")
else:
    print("✅ No hay filas duplicadas")

## 6️⃣ Análisis Univariado

Exploramos la distribución de variables numéricas y categóricas individualmente.

In [ ]:
# Seleccionar solo variables numéricas para estadísticas descriptivas
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

print(f"📊 Estadísticas descriptivas de {len(numeric_cols)} variables numéricas:")
print("="*100)
df[numeric_cols].describe().round(2)

In [ ]:
# Variables categóricas: analizar distribución de valores únicos
categorical_cols = ['Month', 'OperatingSystems', 'Browser', 'Region', 
                    'TrafficType', 'VisitorType', 'Weekend', 'Revenue']

print("📊 Resumen de variables categóricas:")
print("="*100)
for col in categorical_cols:
    print(f"\n{col}:")
    print(f"  Valores únicos: {df[col].nunique()}")
    print(f"  Valores: {df[col].unique()[:10]}")  # Mostrar hasta 10 valores

## 7️⃣ Análisis de la Variable Target (Revenue)

**Crítico**: Análisis del desbalanceo de clases para determinar estrategia de modelado.

In [ ]:
# Análisis de distribución del target (Revenue)
# IMPORTANTE: Determina si necesitamos técnicas de balanceo
revenue_counts = df['Revenue'].value_counts()
revenue_pct = df['Revenue'].value_counts(normalize=True) * 100

print("🎯 Distribución de la variable target (Revenue):")
print("="*80)
print(f"\nFalse (No Compra): {revenue_counts[False]:,} sesiones ({revenue_pct[False]:.2f}%)")
print(f"True (Compra):     {revenue_counts[True]:,} sesiones ({revenue_pct[True]:.2f}%)")
print(f"\n⚖️ Ratio de desbalanceo: {revenue_counts[False]/revenue_counts[True]:.2f}:1")
print(f"⚠️ Dataset desbalanceado - Considerar SMOTE o class weights en modelado")

In [ ]:
# Visualización de Revenue
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Distribución de Revenue', 'Proporción de Revenue'),
    specs=[[{'type': 'bar'}, {'type': 'pie'}]]
)

# Gráfico de barras
fig.add_trace(
    go.Bar(x=revenue_counts.index.astype(str), y=revenue_counts.values,
           marker_color=['#FF6B6B', '#4ECDC4'],
           text=revenue_counts.values,
           textposition='auto'),
    row=1, col=1
)

# Gráfico de pastel
fig.add_trace(
    go.Pie(labels=revenue_counts.index.astype(str), values=revenue_counts.values,
           marker_colors=['#FF6B6B', '#4ECDC4']),
    row=1, col=2
)

fig.update_layout(
    title_text="Análisis de la Variable Target: Revenue",
    showlegend=False,
    height=400
)

fig.show()

## 8️⃣ Análisis de Correlaciones

Identificamos relaciones lineales entre variables numéricas y el target.

In [ ]:
# Seleccionar variables numéricas
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

print(f"📊 Variables Numéricas ({len(numeric_cols)}):")
for i, col in enumerate(numeric_cols, 1):
    print(f"{i:2d}. {col}")

In [ ]:
# Histogramas de todas las variables numéricas
# Permite identificar: distribuciones sesgadas, outliers, multimodalidad
fig, axes = plt.subplots(nrows=5, ncols=2, figsize=(15, 20))
axes = axes.ravel()

for idx, col in enumerate(numeric_cols):
    if idx < len(axes):
        axes[idx].hist(df[col], bins=50, edgecolor='black', alpha=0.7, color='skyblue')
        axes[idx].set_title(f'Distribución de {col}', fontsize=12, fontweight='bold')
        axes[idx].set_xlabel(col)
        axes[idx].set_ylabel('Frecuencia')
        axes[idx].grid(True, alpha=0.3)

# Ocultar ejes sobrantes
for idx in range(len(numeric_cols), len(axes)):
    axes[idx].set_visible(False)

plt.tight_layout()
plt.suptitle('Distribuciones de Variables Numéricas', y=1.001, fontsize=16, fontweight='bold')
plt.show()

print("💡 Observar: Asimetría, concentración en cero, valores extremos")

In [ ]:
# Boxplots para identificar outliers
# Los puntos fuera de los bigotes son potenciales outliers
fig, axes = plt.subplots(nrows=5, ncols=2, figsize=(15, 20))
axes = axes.ravel()

for idx, col in enumerate(numeric_cols):
    if idx < len(axes):
        axes[idx].boxplot(df[col].dropna(), vert=True, patch_artist=True,
                         boxprops=dict(facecolor='lightblue', alpha=0.7),
                         medianprops=dict(color='red', linewidth=2))
        axes[idx].set_title(f'Boxplot de {col}', fontsize=12, fontweight='bold')
        axes[idx].set_ylabel(col)
        axes[idx].grid(True, alpha=0.3)

# Ocultar ejes sobrantes
for idx in range(len(numeric_cols), len(axes)):
    axes[idx].set_visible(False)

plt.tight_layout()
plt.suptitle('Detección de Outliers - Variables Numéricas', y=1.001, fontsize=16, fontweight='bold')
plt.show()

print("💡 Línea roja = Mediana | Puntos = Outliers potenciales")

## 9️⃣ Análisis de Variables Categóricas

Exploramos distribución de categorías y su balance.

In [ ]:
# Seleccionar variables categóricas (excluyendo el target)
categorical_cols = ['Month', 'OperatingSystems', 'Browser', 'Region', 
                    'TrafficType', 'VisitorType', 'Weekend']

print(f"📊 Variables Categóricas ({len(categorical_cols)}):")
for i, col in enumerate(categorical_cols, 1):
    unique_vals = df[col].nunique()
    print(f"{i:2d}. {col:<20} - {unique_vals} valores únicos")

In [ ]:
# Distribución detallada de variables categóricas
# Importante: detectar categorías dominantes o raras
for col in categorical_cols:
    print(f"\n{'='*80}")
    print(f"📊 Distribución de {col}:")
    print(f"{'='*80}")
    
    value_counts = df[col].value_counts()
    value_pct = df[col].value_counts(normalize=True) * 100
    
    result_df = pd.DataFrame({
        'Valor': value_counts.index,
        'Frecuencia': value_counts.values,
        'Porcentaje': value_pct.values
    })
    
    print(result_df.to_string(index=False))
    
    # Visualización interactiva
    fig = px.bar(result_df.head(15), x='Valor', y='Frecuencia',
                 title=f'Top 15 valores de {col}',
                 text='Frecuencia')
    fig.update_traces(texttemplate='%{text:,.0f}', textposition='outside')
    fig.show()

## 🔟 Correlaciones entre Variables Numéricas

In [ ]:
# Matriz de correlación
correlation_matrix = df[numeric_cols].corr()

print("📊 Matriz de Correlación:")
print("="*80)
print(correlation_matrix.round(3))

In [ ]:
# Heatmap de correlaciones
plt.figure(figsize=(14, 10))
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=1,
            cbar_kws={"shrink": 0.8})
plt.title('Matriz de Correlación - Variables Numéricas', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

In [ ]:
# Correlaciones más fuertes (excluyendo diagonal)
correlations = correlation_matrix.abs().unstack()
correlations = correlations[correlations < 1]  # Excluir diagonal
top_correlations = correlations.sort_values(ascending=False).head(10)

print("🔝 Top 10 Correlaciones más fuertes:")
print("="*80)
for (var1, var2), corr in top_correlations.items():
    print(f"{var1:<30} ↔ {var2:<30}: {corr:.3f}")

## 1️⃣1️⃣ Análisis Bivariado: Variables vs Revenue

In [ ]:
# Análisis de variables numéricas por Revenue
for col in numeric_cols:
    fig = go.Figure()
    
    # Compradores
    fig.add_trace(go.Box(
        y=df[df['Revenue']==True][col],
        name='Compra (TRUE)',
        marker_color='#4ECDC4'
    ))
    
    # No compradores
    fig.add_trace(go.Box(
        y=df[df['Revenue']==False][col],
        name='No Compra (FALSE)',
        marker_color='#FF6B6B'
    ))
    
    fig.update_layout(
        title=f'Distribución de {col} por Revenue',
        yaxis_title=col,
        height=400
    )
    
    fig.show()

In [ ]:
# Promedios por Revenue
print("📊 Promedios de Variables Numéricas por Revenue:")
print("="*100)

revenue_stats = df.groupby('Revenue')[numeric_cols].mean().T
revenue_stats.columns = ['No Compra (FALSE)', 'Compra (TRUE)']
revenue_stats['Diferencia %'] = ((revenue_stats['Compra (TRUE)'] - revenue_stats['No Compra (FALSE)']) / 
                                  revenue_stats['No Compra (FALSE)'] * 100)

revenue_stats.style.background_gradient(cmap='RdYlGn', subset=['Diferencia %'])

In [ ]:
# Análisis de variables categóricas por Revenue
for col in ['Month', 'VisitorType', 'Weekend']:
    crosstab = pd.crosstab(df[col], df['Revenue'], normalize='index') * 100
    
    fig = go.Figure()
    
    fig.add_trace(go.Bar(
        name='No Compra',
        x=crosstab.index,
        y=crosstab[False],
        marker_color='#FF6B6B'
    ))
    
    fig.add_trace(go.Bar(
        name='Compra',
        x=crosstab.index,
        y=crosstab[True],
        marker_color='#4ECDC4'
    ))
    
    fig.update_layout(
        title=f'Tasa de Conversión por {col}',
        xaxis_title=col,
        yaxis_title='Porcentaje (%)',
        barmode='stack',
        height=400
    )
    
    fig.show()

## 1️⃣2️⃣ Detección de Outliers

In [ ]:
# Detección de outliers usando IQR
print("🔍 Detección de Outliers (Método IQR):")
print("="*80)

outlier_summary = []

for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    n_outliers = len(outliers)
    pct_outliers = (n_outliers / len(df)) * 100
    
    outlier_summary.append({
        'Variable': col,
        'Q1': Q1,
        'Q3': Q3,
        'IQR': IQR,
        'Lower Bound': lower_bound,
        'Upper Bound': upper_bound,
        'N° Outliers': n_outliers,
        '% Outliers': pct_outliers
    })

outlier_df = pd.DataFrame(outlier_summary)
outlier_df = outlier_df.sort_values('N° Outliers', ascending=False)
print(outlier_df.to_string(index=False))

## 1️⃣3️⃣ Insights y Conclusiones

In [ ]:
print("📝 RESUMEN DEL ANÁLISIS EXPLORATORIO")
print("="*100)

print("\n✅ CALIDAD DE DATOS:")
print(f"  - Total de registros: {len(df):,}")
print(f"  - Valores faltantes: {df.isnull().sum().sum()}")
print(f"  - Duplicados: {df.duplicated().sum()}")

print("\n📊 VARIABLE TARGET (Revenue):")
print(f"  - Tasa de conversión: {(df['Revenue'].sum() / len(df) * 100):.2f}%")
print(f"  - Balance de clases: {revenue_counts[False] / revenue_counts[True]:.2f}:1 (Desbalanceado)")

print("\n🔑 VARIABLES NUMÉRICAS:")
print(f"  - Total: {len(numeric_cols)}")
print(f"  - Variables con outliers significativos (>5%): {len(outlier_df[outlier_df['% Outliers'] > 5])}")

print("\n🏷️ VARIABLES CATEGÓRICAS:")
print(f"  - Total: {len(categorical_cols)}")
print(f"  - Month: {df['Month'].nunique()} meses")
print(f"  - VisitorType: {df['VisitorType'].nunique()} tipos")

print("\n💡 PRINCIPALES INSIGHTS:")
print("  1. Dataset desbalanceado: ~85% No compra vs ~15% Compra")
print("  2. Variables de duración tienen alta variabilidad")
print("  3. PageValues muestra clara diferencia entre compradores y no compradores")
print("  4. Visitors que regresan (Returning_Visitor) tienen mayor tasa de conversión")
print("  5. Presencia de outliers en variables de duración y páginas visitadas")

print("\n🚀 PRÓXIMOS PASOS:")
print("  1. Preprocesamiento: Manejo de outliers")
print("  2. Feature Engineering: Crear variables derivadas (ej: tiempo promedio por página)")
print("  3. Encoding de variables categóricas")
print("  4. Balanceo de clases (SMOTE o undersampling)")
print("  5. Selección de features")

---

## 🎯 Conclusiones del EDA

### ✅ Calidad del Dataset
- **Sin valores faltantes**: Dataset limpio, no requiere imputación
- **Sin duplicados**: Cada sesión es única
- **Tipos de datos apropiados**: Correctamente tipificados

### ⚠️ Desafíos Identificados

1. **Desbalanceo severo de clases**
   - No Compra: 84.5% (10,422 sesiones)
   - Compra: 15.5% (1,908 sesiones)
   - **Ratio**: 5.4:1
   - **Solución**: SMOTE o class weights en modelado

2. **Outliers en variables continuas**
   - Duraciones con valores extremos (>99 percentil)
   - BounceRates y ExitRates con picos en 0.2
   - **Solución**: Winsorización (percentiles 1-99)

3. **Escalas diferentes entre variables**
   - PageValues: 0-361
   - Duraciones: 0-65,000 segundos
   - **Solución**: StandardScaler

### 💡 Variables Más Prometedoras

1. **PageValues** (⭐⭐⭐⭐⭐)
   - Mayor poder discriminante
   - Compradores tienen PageValues significativamente más altos

2. **ProductRelated_Duration** (⭐⭐⭐⭐)
   - Compradores pasan más tiempo en páginas de productos
   - Correlación positiva con Revenue

3. **ExitRates** (⭐⭐⭐⭐)
   - Compradores tienen menores tasas de salida
   - Correlación negativa con Revenue

4. **VisitorType** (⭐⭐⭐)
   - Returning_Visitor tiene mayor tasa de conversión
   - New_Visitor menos propenso a comprar

### 🔜 Próximos Pasos

**Notebook 02_preprocesamiento_dataset.ipynb**:
1. Feature Engineering (crear variables derivadas)
2. Encoding de variables categóricas
3. Manejo de outliers (winsorización)
4. Escalado con StandardScaler
5. Balanceo de clases con SMOTE
6. División train/test estratificada

---

**📊 Dataset**: UCI ML Repository | **Autores**: C. Sakar, Y. Kastro | **DOI**: 10.24432/C5F88Q